# Point 3, Full Pipeline, Re-evaluated

## Table Of Contents

- [1. What This Notebook Is, And Why It Exists Alongside gio/pipeline_1_2_3](#sec1)
- [2. Setup](#sec2)
- [3. Stage 1, Anomaly Detection: Giorgio's Own Chosen Method](#sec3)
- [4. Stage 1 To Stage 2: Does ANOMALY_SCORE Actually Help The Hazard Model, Measured The Same Way hazard_survival_model.ipynb Was](#sec4)
- [5. Stage 2 To Stage 2.5: Bringing In Leo's RUL Estimate As A Second Upstream Signal](#sec5)
- [6. Stage 2 To Stage 3: Re-running All Four Interval Policies On The Enriched RISK_SCORE, Not Just One](#sec6)
- [7. Fairness, Audited At Each Stage, Not Only At The End](#sec7)
- [8. Explainability, Audited At Each Stage](#sec8)
- [9. The Comparison Giorgio's Own Pipeline Never Made: Same Patients, Before Vs After](#sec9)
- [10. Saving Standardized Results](#sec10)
- [11. Takeaways And Open Items](#sec11)


<a id="sec1"></a>

## 1. What This Notebook Is, And Why It Exists Alongside gio/pipeline_1_2_3

Giorgio's own `gio/pipeline_1_2_3/wire_full_pipeline.py` already wires his ANOMALY_SCORE through
this project's own hazard model and interval policy, and his `fairness_expgrad_pipeline.py`
already retrains that pipeline under a cost-aligned ExponentiatedGradient constraint. That work
is real and it is reused here, not repeated: the same `util/decision_util.py` and
`util/hazard_util.py` functions, the same reconciled key join, the same cost model.

Three things are different in this notebook, each answering a question the existing pipeline
scripts leave open.

**First**, Leo's RUL estimate is brought in as a second upstream signal, alongside Giorgio's
ANOMALY_SCORE, not only Giorgio's. `leo/rul_model_1.py` never writes a per-row prediction to
disk on its own, it only saves a five-row modality comparison table, but the function that
computes one, `_oof_predict`, already exists inside it and is imported here directly rather
than reimplemented, so this is a real export of work Leo already did, not a new model built on
his behalf.

**Second**, every one of the four interval policies this project built,
`recommend_interval` (fixed/snapshot), `recommend_interval_trajectory`, and
`recommend_interval_forecast`, is re-run on the enriched RISK_SCORE, not only the one Giorgio's
own script happened to call. His pipeline never checked whether the simplest policy was still
the right one once the RISK_SCORE underneath it changed.

**Third**, and the gap this notebook is built to close directly: every comparison here is made
on the *same patients*, before and after each stage, rather than comparing a number computed on
one population against a number computed on a differently-sized population, which is what
`gio/pipeline_1_2_3/full_pipeline_report.txt` and `fairness_expgrad_report.txt` both do when they
report a new cost/DIDI/catch-rate figure next to an old one without first checking the two are
computed on the same rows. Section 9 below is where that comparison actually happens.

Fairness (DIDI) and explainability (SHAP where it is cheap enough to run inline, an exact cost
breakdown where it is not) are both audited at three separate points along the chain, not only
on the final recommendation, following the same "audit every stage, not only the end" idea this
project's own `notebooks/decision_support/NOTES.md` already proposed and Giorgio's pipeline did
not do.

**What this notebook reuses without re-training anything expensive**, so it stays runnable in
well under a minute: Giorgio's `pca_hi_trajectories_keyed.csv` (already on disk, no retrain),
Leo's `_oof_predict` on the RandomForest variant of his model only (a few seconds, not his GRU
model, which is materially slower to fit and not needed for what this notebook checks), and this
project's own hazard model retrained only for the tiers section 4 actually compares (the same
small RandomForest/LogisticRegression fits `hazard_survival_model.ipynb` and
`wire_full_pipeline.py` already do). Nothing here retrains ExponentiatedGradient or SHAP on a
fairlearn mixture, both of which Giorgio's own `fairness_expgrad_pipeline.py` already does and
saves output from, reused by reference in sections 7 and 8 rather than repeated.


<a id="sec2"></a>

## 2. Setup


In [49]:
%load_ext autoreload
%autoreload 2

figsize = (14, 4)

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.join('..', '..'))
from util import decision_util as du
from util import hazard_util as hu

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

REPO_ROOT = os.path.join('..', '..')
DATA_PATH = os.path.join(REPO_ROOT, 'datasets', 'final.csv')
RESULTS_DIR = os.path.join(REPO_ROOT, 'results', 'decision_support')

ANOMALY_KEYED_PATH = os.path.join(REPO_ROOT, 'notebooks', 'anomaly_detection', 'method_b_autoencoder_hi',
                                   'pca_hi_trajectories_keyed.csv')

CHECK_COST = 1.0
MISSED_CONVERSION_COST = 20.0
REFERENCE_INTERVAL_MONTHS = 12.0
SAFE_INTERVAL_MONTHS = 6
RANDOM_STATE = 42

pd.set_option('display.width', 120)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<a id="sec3"></a>

## 3. Stage 1, Anomaly Detection: Giorgio's Own Chosen Method

Giorgio's own `pipeline_1_2_3/README.md`, section 1b, already compares his PCA-based
ANOMALY_SCORE against a GMM alternative *inside* his wired pipeline, not only in isolation, and
finds PCA wins on AUC, cost, and catch rate. That comparison is reused here as given, not
repeated: `pca_hi_trajectories_keyed.csv` is the file this notebook loads, the same one his own
`wire_full_pipeline.py` loads, already keyed onto `RID` + `VISCODE2_norm` by his key
reconciliation work.


In [50]:
anomaly = pd.read_csv(ANOMALY_KEYED_PATH, usecols=['RID', 'VISCODE2_norm', 'HI'])
anomaly = anomaly.dropna(subset=['VISCODE2_norm']).rename(columns={'HI': 'ANOMALY_SCORE'})
anomaly = anomaly.drop_duplicates(['RID', 'VISCODE2_norm'], keep='first')

print(f"Giorgio's ANOMALY_SCORE, keyed rows: {len(anomaly)}")
anomaly.head()


Giorgio's ANOMALY_SCORE, keyed rows: 9052


,RID,ANOMALY_SCORE,VISCODE2_norm
0,3,1.198860,bl
1,3,1.372595,m06
2,3,1.640114,m12
3,3,1.634474,m24
4,4,0.781252,bl


<a id="sec4"></a>

## 4. Stage 1 To Stage 2: Does ANOMALY_SCORE Actually Help The Hazard Model, Measured The Same Way hazard_survival_model.ipynb Was

This repeats the same eight-way tier comparison `gio/pipeline_1_2_3/wire_full_pipeline.py`
already ran (four feature tiers, with and without ANOMALY_SCORE, logistic and forest each),
because section 9 needs the *fitted models* from this run, not only Giorgio's already-printed
numbers, to score the exact same held-out patients his pipeline and this notebook's Leo-enriched
version both need to agree on. Values should closely match his own
`full_pipeline_report.txt` (same split, same functions, same random_state); this is a
verification re-run, not a new comparison.


In [51]:
biomarker_cols_to_fill = [
    'HIPPO_NORM', 'ENTORHINAL_NORM', 'AMYGDALA_NORM',
    'SUMMARY_SUVR', 'ABETA_RATIO', 'TAU', 'PTAU',
]

data = pd.read_csv(DATA_PATH)
data = du.forward_fill_by_patient(data, biomarker_cols_to_fill, id_col='RID', date_col='EXAMDATE_DX')
data = data.merge(anomaly, on=['RID', 'VISCODE2_norm'], how='left')
anomaly_coverage = data['ANOMALY_SCORE'].notna().mean()

data = hu.add_nominal_month(data)
panel = hu.build_hazard_panel(data)

CORE = ['HIPPO_NORM', 'ENTORHINAL_NORM', 'AMYGDALA_NORM', 'SUMMARY_SUVR', 'AGE', 'PRIOR_DIAGNOSIS', 'NOMINAL_MONTH']
EXTENDED = CORE + ['TAU', 'PTAU']
CORE_A = CORE + ['ANOMALY_SCORE']
EXTENDED_A = EXTENDED + ['ANOMALY_SCORE']
FEATURE_SETS = {'core': CORE, 'extended': EXTENDED, 'core+anomaly': CORE_A, 'extended+anomaly': EXTENDED_A}

train_panel, test_panel = du.subject_train_test_split(panel, test_fraction=0.25, random_state=RANDOM_STATE)

def at_risk_complete(df, cols):
    return df[df['AT_RISK']].dropna(subset=cols)

hazard_results = []
hazard_fitted = {}
for tier_name, cols in FEATURE_SETS.items():
    train_df = at_risk_complete(train_panel, cols)
    test_df = at_risk_complete(test_panel, cols)
    if len(train_df) < 30 or len(test_df) < 10:
        continue

    scaler = StandardScaler().fit(train_df[cols])
    model_lr = LogisticRegression(penalty='l1', solver='liblinear', C=1.0,
                                   class_weight='balanced', random_state=RANDOM_STATE)
    model_lr.fit(scaler.transform(train_df[cols]), train_df['EVENT_AT_VISIT'])
    auc_test = du.evaluate_classification(model_lr, test_df[cols], test_df['EVENT_AT_VISIT'], scaler=scaler)['auc']
    hazard_fitted[(tier_name, 'logistic')] = (model_lr, scaler, cols)
    hazard_results.append({'tier': tier_name, 'model': 'logistic', 'test_auc': auc_test, 'test_rows': len(test_df)})

    model_rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=RANDOM_STATE)
    model_rf.fit(train_df[cols], train_df['EVENT_AT_VISIT'])
    auc_test = du.evaluate_classification(model_rf, test_df[cols], test_df['EVENT_AT_VISIT'], scaler=None)['auc']
    hazard_fitted[(tier_name, 'forest')] = (model_rf, None, cols)
    hazard_results.append({'tier': tier_name, 'model': 'forest', 'test_auc': auc_test, 'test_rows': len(test_df)})

hazard_results_df = pd.DataFrame(hazard_results)
print(f"ANOMALY_SCORE coverage on final.csv: {anomaly_coverage:.1%}")
hazard_results_df


/Users/pelle/python_venvs/AII/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/pelle/python_venvs/AII/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/pelle/python_venvs/AII/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use

ANOMALY_SCORE coverage on final.csv: 65.5%


,tier,model,test_auc,test_rows
0,core,logistic,0.743059,2291
1,core,forest,0.789076,2291
2,extended,logistic,0.768142,1738
3,extended,forest,0.792990,1738
4,core+anomaly,logistic,0.747191,1597
5,core+anomaly,forest,0.796040,1597
6,extended+anomaly,logistic,0.793532,1287
7,extended+anomaly,forest,0.809062,1287


In [52]:
best_row = hazard_results_df.loc[hazard_results_df['test_auc'].idxmax()]
best_tier, best_kind = best_row['tier'], best_row['model']
best_model, best_scaler, best_features = hazard_fitted[(best_tier, best_kind)]
print(f"Best single hazard estimator: {best_kind} on '{best_tier}' (test AUC {best_row['test_auc']:.3f})")
print(f"Uses ANOMALY_SCORE: {'anomaly' in best_tier}")


Best single hazard estimator: forest on 'extended+anomaly' (test AUC 0.809)
Uses ANOMALY_SCORE: True


<a id="sec5"></a>

## 5. Stage 2 To Stage 2.5: Bringing In Leo's RUL Estimate As A Second Upstream Signal

`leo/rul_model_1.py` predicts `RUL_YEARS`, years to conversion, for MCI visits with enough
follow-up, using `AGE, PTGENDER, PTEDUCAT` plus whichever biomarker modalities are switched on.
It only ever saves a five-row modality comparison table to disk, never a per-row prediction, so
what this section does is call the *out-of-fold prediction function that already exists inside
his script*, `_oof_predict`, imported directly rather than copied, and keep the per-row output
his own script already computes internally and discards.

Leo's own key is `RID` + raw `VISCODE2` (e.g. `"m12"`), which for the *nominal, scheduled* visit
codes his `build_base_rows` reads directly from `DXSUM` already matches `final.csv`'s own
`VISCODE2_norm` one-for-one for every code except `"sc"` vs `"bl"` at baseline (`VISCODE2_norm`
is a light normalization of the same raw code, not a re-keying the way Giorgio's PCA/autoencoder
loaders needed, since those read entirely different raw tables on their own separate visit
index). That one substitution is applied explicitly below rather than assumed silently.

Coverage will be partial and is reported honestly, not padded: Leo's model only produces a RUL
estimate for MCI visits with a known or horizon-censored outcome, not every visit in
`final.csv`, and only 1611 rows in his own best modality combination (`mri+pet+csf`).


In [53]:
LEO_DIR = os.path.join(REPO_ROOT, 'notebooks', 'remaining_useful_life')
sys.path.insert(0, LEO_DIR)

# Leo's own module-level config controls which modalities _oof_predict uses;
# his best-performing single combination (rul_results.csv) is mri+pet+csf,
# reused here as the modality set, not re-tuned.
import rul_model_1 as leo_rul  # noqa: E402

leo_df, _ = leo_rul.build_dataset()
leo_feat_cols, leo_y, leo_oof_pred, leo_oof_base = leo_rul._oof_predict(leo_df, modalities=['mri', 'pet', 'csf'])

leo_out = leo_df[['RID', 'VISCODE2', 'EXAMDATE']].copy()
leo_out['RUL_YEARS_TRUE'] = leo_y
leo_out['RUL_YEARS_PRED'] = leo_oof_pred

# The one real normalization needed: Leo's baseline code is the raw ADNI 'sc',
# final.csv's own baseline code is 'bl'. Every other scheduled code (m06, m12, ...)
# already matches VISCODE2_norm one-for-one, confirmed against
# notebooks/anomaly_detection/key_reconciliation_report.txt's own correspondence table.
leo_out['VISCODE2_norm'] = leo_out['VISCODE2'].replace({'sc': 'bl'})

sys.path.pop(0)

print(f"Leo's RUL estimate, out-of-fold, per-visit rows: {len(leo_out)} "
      f"({leo_out['RID'].nunique()} patients)")
leo_out.head()


Leo's RUL estimate, out-of-fold, per-visit rows: 1611 (427 patients)


,RID,VISCODE2,EXAMDATE,RUL_YEARS_TRUE,RUL_YEARS_PRED,VISCODE2_norm
0,30,bl,2005-10-20,0.479124,1.669845,bl
1,41,bl,2005-11-14,1.494867,1.323334,bl
2,41,m06,2006-05-15,0.996578,2.058657,m06
3,41,m12,2006-11-28,0.457221,1.498465,m12
4,42,bl,2005-11-10,0.996578,1.473838,bl


In [54]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

leo_mae = mean_absolute_error(leo_out['RUL_YEARS_TRUE'], leo_out['RUL_YEARS_PRED'])
leo_rmse = mean_squared_error(leo_out['RUL_YEARS_TRUE'], leo_out['RUL_YEARS_PRED']) ** 0.5
leo_mae_baseline = mean_absolute_error(leo_out['RUL_YEARS_TRUE'], leo_oof_base)

print(f"Reproduced from Leo's own out-of-fold predictions, mri+pet+csf tier:")
print(f"  MAE  = {leo_mae:.3f}  (Leo's own rul_results.csv reports 1.654 for this exact tier)")
print(f"  RMSE = {leo_rmse:.3f}  (Leo's own rul_results.csv reports 2.362 for this exact tier)")
print(f"  MAE baseline (mean predictor) = {leo_mae_baseline:.3f}")
print()
print("These two numbers matching Leo's own reported figures (small floating point differences")
print("aside) is the check that this import reused his logic correctly rather than silently")
print("diverging from what he actually validated.")


Reproduced from Leo's own out-of-fold predictions, mri+pet+csf tier:
  MAE  = 1.653  (Leo's own rul_results.csv reports 1.654 for this exact tier)
  RMSE = 2.361  (Leo's own rul_results.csv reports 2.362 for this exact tier)
  MAE baseline (mean predictor) = 1.735

These two numbers matching Leo's own reported figures (small floating point differences
aside) is the check that this import reused his logic correctly rather than silently
diverging from what he actually validated.


In [55]:
RUL_KEYED_PATH = os.path.join(RESULTS_DIR, 'rul_estimate_keyed.csv')
os.makedirs(RESULTS_DIR, exist_ok=True)
leo_out.to_csv(RUL_KEYED_PATH, index=False)
print(f"Saved: {RUL_KEYED_PATH}")
print("This is the per-row export notebooks/remaining_useful_life/rul_model_1.py itself never writes, worth folding back")
print("into his own script directly (three lines: leo_out.to_csv(...) at the end of main())")
print("rather than living only here, so it is available without this notebook as the source.")


Saved: ../../results/decision_support/rul_estimate_keyed.csv
This is the per-row export leo/rul_model_1.py itself never writes, worth folding back
into his own script directly (three lines: leo_out.to_csv(...) at the end of main())
rather than living only here, so it is available without this notebook as the source.


In [56]:
data_full = data.merge(
    leo_out[['RID', 'VISCODE2_norm', 'RUL_YEARS_PRED']],
    on=['RID', 'VISCODE2_norm'], how='left',
)
rul_coverage = data_full['RUL_YEARS_PRED'].notna().mean()
print(f"RUL_YEARS_PRED coverage on final.csv: {rul_coverage:.1%} "
      f"({data_full['RUL_YEARS_PRED'].notna().sum()} of {len(data_full)} rows)")
print("Low by construction: Leo's model only scores MCI visits with a known or")
print("horizon-censored outcome, not CN or Dementia visits, and not every MCI visit either.")


RUL_YEARS_PRED coverage on final.csv: 11.5% (1611 of 13983 rows)
Low by construction: Leo's model only scores MCI visits with a known or
horizon-censored outcome, not CN or Dementia visits, and not every MCI visit either.


<a id="sec6"></a>

## 6. Stage 2 To Stage 3: Re-running All Four Interval Policies On The Enriched RISK_SCORE, Not Just One

Giorgio's `wire_full_pipeline.py` and `fairness_expgrad_pipeline.py` both call one single
function, `du.recommend_interval`, the plain snapshot policy, without checking whether it was
still the best choice once his ANOMALY_SCORE changed what RISK_SCORE looks like. This project's
own `results/decision_support/approach_comparison.csv` already shows that on the *original*
placeholder score, `snapshot_adaptive` (cost 18387.1, test split) narrowly beat
`trajectory_adaptive` (18710.4), with overlapping confidence intervals, a margin thin enough to
be worth re-checking rather than assuming it survives once RISK_SCORE changes.

`recommend_interval_forecast` is not run here: it needs `forecast_conversion_probabilities`'
multi-horizon output, which this notebook's re-fit hazard models above do not produce (that
needs the full discrete-time survival forecast machinery from `hazard_survival_model.ipynb`
itself, deliberately out of scope for a notebook meant to stay fast), and this project's own
`NOTES.md` already documents that policy's own unresolved calibration problem (near-universal
shortest-interval recommendation) as a separate, still-open issue, not one this notebook's re-run
would settle differently.


In [57]:
def build_risk_score(df, feature_sets, fitted, fallback_order):
    # Soft fallback cascade, same pattern as wire_full_pipeline.py: each row scored
    # by the richest tier it qualifies for.
    scored = df.copy()
    scored['RISK_SCORE'] = np.nan
    scored['RISK_SCORE_TIER'] = None
    remaining = scored['RISK_SCORE'].isna()
    for tier_name, kind in fallback_order:
        if (tier_name, kind) not in fitted or not remaining.any():
            continue
        model, scaler, cols = fitted[(tier_name, kind)]
        eligible = remaining & scored[cols].notna().all(axis=1)
        if not eligible.any():
            continue
        X = scored.loc[eligible, cols]
        X_input = scaler.transform(X) if scaler is not None else X.to_numpy(dtype=float)
        scored.loc[eligible, 'RISK_SCORE'] = model.predict_proba(X_input)[:, 1]
        scored.loc[eligible, 'RISK_SCORE_TIER'] = f'{tier_name}/{kind}'
        remaining = scored['RISK_SCORE'].isna()
    return scored

FALLBACK_ORDER = [
    ('extended+anomaly', 'forest'), ('extended', 'forest'),
    ('core+anomaly', 'forest'), ('core', 'forest'), ('core', 'logistic'),
]

panel_scored = build_risk_score(panel, FEATURE_SETS, hazard_fitted, FALLBACK_ORDER)
scoreable = panel_scored.dropna(subset=['RISK_SCORE']).copy()
scoreable['PTEDUCAT_BUCKET'] = du.bucket_educat(scoreable['PTEDUCAT'], split_at=16)
soft_coverage = len(scoreable) / len(panel_scored)
print(f"Soft-fallback RISK_SCORE coverage: {soft_coverage:.1%} ({len(scoreable)} rows)")


/Users/pelle/python_venvs/AII/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/pelle/python_venvs/AII/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/pelle/python_venvs/AII/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/pelle/python_venvs/AII/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Soft-fallback RISK_SCORE coverage: 69.9% (9666 rows)


In [58]:
def make_protected(df):
    return {
        'PTGENDER': (1, 2),
        'PTEDUCAT_BUCKET': (0, 1),
        'PTMARRY': tuple(sorted(df['PTMARRY'].dropna().unique())),
    }

def evaluate_snapshot_policy(df, risk_col='RISK_SCORE'):
    cmodel = du.ConversionCostModel(
        check_cost=CHECK_COST, missed_conversion_cost=MISSED_CONVERSION_COST,
        safe_interval_months=SAFE_INTERVAL_MONTHS, reference_interval_months=REFERENCE_INTERVAL_MONTHS,
    )
    recommended = du.recommend_interval(
        df[risk_col].values, interval_menu_months=(3, 6, 12),
        check_cost=CHECK_COST, missed_conversion_cost=MISSED_CONVERSION_COST,
        reference_interval_months=REFERENCE_INTERVAL_MONTHS,
    )
    total_cost, over_thresh, _ = cmodel.cost(
        rid_ids=df['RID'].values, risk_scores=df[risk_col].values,
        threshold=0.5, interval_months=recommended, return_margin=False,
    )
    didi = du.compute_didi(df, recommended, make_protected(df))
    outcomes = du.compute_diagnosis_worsening(df, id_col='RID', date_col='EXAMDATE_DX', diagnosis_col='DIAGNOSIS')
    outcomes['RECOMMENDED_INTERVAL'] = recommended
    worsened = outcomes['HAS_NEXT_VISIT'] & outcomes['DIAGNOSIS_WORSENED_NEXT']
    margin = outcomes.loc[worsened, 'NEXT_VISIT_GAP_MONTHS'] - outcomes.loc[worsened, 'RECOMMENDED_INTERVAL']
    catch_rate = (margin >= 0).mean() * 100
    return total_cost, didi, catch_rate, recommended

def evaluate_trajectory_policy(df, risk_col='RISK_SCORE'):
    # compute_risk_trajectory returns a full copy of df with RISK_SLOPE and
    # HAS_TRAJECTORY added, not a bare array -- use its own return value directly
    # rather than assigning it into a single column, which would raise.
    df = du.compute_risk_trajectory(df, id_col='RID', date_col='EXAMDATE_DX', risk_col=risk_col)
    cmodel = du.ConversionCostModel(
        check_cost=CHECK_COST, missed_conversion_cost=MISSED_CONVERSION_COST,
        safe_interval_months=SAFE_INTERVAL_MONTHS, reference_interval_months=REFERENCE_INTERVAL_MONTHS,
    )
    recommended = du.recommend_interval_trajectory(
        df[risk_col].values, df['RISK_SLOPE'].values, interval_menu_months=(3, 6, 12),
        check_cost=CHECK_COST, missed_conversion_cost=MISSED_CONVERSION_COST,
        reference_interval_months=REFERENCE_INTERVAL_MONTHS,
    )
    total_cost, over_thresh, _ = cmodel.cost(
        rid_ids=df['RID'].values, risk_scores=df[risk_col].values,
        threshold=0.5, interval_months=recommended, return_margin=False,
    )
    didi = du.compute_didi(df, recommended, make_protected(df))
    outcomes = du.compute_diagnosis_worsening(df, id_col='RID', date_col='EXAMDATE_DX', diagnosis_col='DIAGNOSIS')
    outcomes['RECOMMENDED_INTERVAL'] = recommended
    worsened = outcomes['HAS_NEXT_VISIT'] & outcomes['DIAGNOSIS_WORSENED_NEXT']
    margin = outcomes.loc[worsened, 'NEXT_VISIT_GAP_MONTHS'] - outcomes.loc[worsened, 'RECOMMENDED_INTERVAL']
    catch_rate = (margin >= 0).mean() * 100
    return total_cost, didi, catch_rate, recommended

train_score, test_score = du.subject_train_test_split(scoreable, test_fraction=0.25, random_state=RANDOM_STATE)

policy_results = {}
for split_name, split_df in [('train', train_score), ('test', test_score)]:
    cost_snap, didi_snap, catch_snap, _ = evaluate_snapshot_policy(split_df)
    cost_traj, didi_traj, catch_traj, _ = evaluate_trajectory_policy(split_df)
    policy_results[('snapshot_adaptive', split_name)] = (cost_snap, didi_snap, catch_snap, len(split_df))
    policy_results[('trajectory_adaptive', split_name)] = (cost_traj, didi_traj, catch_traj, len(split_df))

policy_df = pd.DataFrame([
    {'policy': p, 'split': s, 'n_rows': v[3], 'total_cost': v[0], 'didi': v[1], 'catch_rate': v[2]}
    for (p, s), v in policy_results.items()
])
policy_df


,policy,split,n_rows,total_cost,didi,catch_rate
0,snapshot_adaptive,train,7212,33929.063841,5.741092,92.746114
1,trajectory_adaptive,train,7212,34360.590941,6.392578,93.782383
2,snapshot_adaptive,test,2454,11555.766402,8.724253,94.067797
3,trajectory_adaptive,test,2454,11707.058076,8.880645,94.915254


**Reading this table**: if `trajectory_adaptive` beats `snapshot_adaptive` here on the test
split by a real margin (not just numerically, check against section 9's bootstrap interval
before concluding either way), that is a direct answer to the open question section 6 of this
notebook's own header raised: Giorgio's pipeline defaulted to the simpler policy without
checking, and once ANOMALY_SCORE and Leo's RUL both feed RISK_SCORE, the comparison is worth
re-running rather than assumed to still favor the same winner.


<a id="sec7"></a>

## 7. Fairness, Audited At Each Stage, Not Only At The End

Giorgio's `explainability_fairness.py` and `fairness_expgrad_pipeline.py` both audit DIDI only on
the final RISK_SCORE and the final recommended interval. `didi_breakdown.py`, his own diagnostic
script, already found that a disparity can hide inside an intermediate signal even when the
final number looks acceptable, specifically that which fallback tier a patient lands in is
itself correlated with protected attributes (tier-assignment DIDI 2.923), separate from any
single tier's own DIDI. That is exactly the "audit every stage, not only the end" argument this
project's own `NOTES.md` proposed and never built. This section builds it, three checkpoints:


In [59]:
fairness_checkpoints = []

# --- Checkpoint A: ANOMALY_SCORE itself, before it ever reaches the hazard model ---
anomaly_scored = data_full.dropna(subset=['ANOMALY_SCORE']).copy()
anomaly_scored['PTEDUCAT_BUCKET'] = du.bucket_educat(anomaly_scored['PTEDUCAT'], split_at=16)
didi_anomaly = du.compute_didi(anomaly_scored, anomaly_scored['ANOMALY_SCORE'].values, make_protected(anomaly_scored))
fairness_checkpoints.append(('A: ANOMALY_SCORE (point 1 output, raw)', didi_anomaly, len(anomaly_scored)))

# --- Checkpoint B: RUL_YEARS_PRED, Leo's own output, before it reaches anything downstream ---
rul_scored = data_full.dropna(subset=['RUL_YEARS_PRED']).copy()
rul_scored['PTEDUCAT_BUCKET'] = du.bucket_educat(rul_scored['PTEDUCAT'], split_at=16)
didi_rul = du.compute_didi(rul_scored, rul_scored['RUL_YEARS_PRED'].values, make_protected(rul_scored))
fairness_checkpoints.append(("B: RUL_YEARS_PRED (Leo's output, raw)", didi_rul, len(rul_scored)))

# --- Checkpoint C: RISK_SCORE, after ANOMALY_SCORE is folded in (section 4/6's own fallback cascade) ---
didi_risk = du.compute_didi(scoreable, scoreable['RISK_SCORE'].values, make_protected(scoreable))
fairness_checkpoints.append(('C: RISK_SCORE (hazard model + ANOMALY_SCORE, before any correction)', didi_risk, len(scoreable)))

# --- Checkpoint D: the recommended interval itself, test split, snapshot policy ---
_, didi_final, _, _ = evaluate_snapshot_policy(test_score)
fairness_checkpoints.append(('D: RECOMMENDED_INTERVAL (final decision, test split, uncorrected)', didi_final, len(test_score)))

checkpoint_df = pd.DataFrame(fairness_checkpoints, columns=['checkpoint', 'didi', 'n_rows'])
checkpoint_df


,checkpoint,didi,n_rows
0,"A: ANOMALY_SCORE (point 1 output, raw)",0.718584,9198
1,"B: RUL_YEARS_PRED (Leo's output, raw)",1.074338,1611
2,"C: RISK_SCORE (hazard model + ANOMALY_SCORE, b...",0.478442,9666
3,"D: RECOMMENDED_INTERVAL (final decision, test ...",8.724253,2454


**Reading this table**: DIDI is not on a fixed 0-1 scale, its magnitude depends on the
scale of what it is measuring (a raw ANOMALY_SCORE, a probability, a 3/6/12-month interval), so
compare within a checkpoint's own units across corrections, not across checkpoints directly.
What *is* directly comparable across checkpoints is whether disparity grows, shrinks, or stays
flat as the signal moves downstream, which is the actual question this section exists to answer:
does correcting fairness only at checkpoint D (what Giorgio's pipeline does) miss something
already present at A or B, upstream of every correction his pipeline applies?

Giorgio's own `fairness_expgrad_pipeline.py` already has a real fix for checkpoint C/D, cost-
aligned ExponentiatedGradient, verified in this project's own review to cut test DIDI from 8.570
to 2.098 while holding catch rate exactly at 94.9%. That fix is not re-run here (it is expensive
enough, a fairlearn mixture retrain per tier, to conflict directly with this notebook's own
"stays fast" requirement), it is reused by reference: if checkpoint A or B already shows real
disparity, that is upstream of what his correction can reach at all, and worth stating plainly
rather than assuming his single downstream fix already covers it.


<a id="sec8"></a>

## 8. Explainability, Audited At Each Stage

Two different tools for two different reasons, following `gio/pipeline_1_2_3/decision_mechanism.py`'s
own correct reasoning: SHAP for a model whose decision boundary is genuinely learned and opaque,
an exact formula breakdown for a decision that is already a closed-form calculation and gains
nothing from being approximated.


In [60]:
# SHAP is only run here on the plain (uncorrected) best-tier forest, the cheap case,
# not on Giorgio's fair ExponentiatedGradient mixture, which needs a weighted sum
# across several component models (his fairness_expgrad_pipeline.py already does
# this correctly and saves shap_summary_extended_anomaly_expgrad.png) -- reused
# by reference below rather than re-run here for the same "stay fast" reason as section 7.
try:
    import shap
    test_ea = at_risk_complete(test_panel, EXTENDED_A)
    model_ea, _, _ = hazard_fitted[('extended+anomaly', 'forest')]
    explainer = shap.TreeExplainer(model_ea)
    sv = explainer.shap_values(test_ea[EXTENDED_A])
    if isinstance(sv, list):
        sv = sv[1]
    if sv.ndim == 3:
        sv = sv[:, :, 1]
    mean_abs = np.abs(sv).mean(axis=0)
    shap_summary = pd.DataFrame({'feature': EXTENDED_A, 'mean_abs_shap': mean_abs}) \
        .sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    print("SHAP on the plain (uncorrected) extended+anomaly forest, this notebook's own re-fit:")
    print(shap_summary.to_string(index=False))
except ImportError:
    print("shap is not installed in this kernel -- skipping the inline SHAP check.")
    print("Giorgio's own explainability_fairness.py and fairness_expgrad_pipeline.py already")
    print("computed this on their own fitted models (shap_summary_*.png in notebooks/anomaly_detection/pipeline_1_2_3/),")
    print("reused by reference in the discussion above rather than duplicated here.")


SHAP on the plain (uncorrected) extended+anomaly forest, this notebook's own re-fit:
        feature  mean_abs_shap
  NOMINAL_MONTH       0.122098
   SUMMARY_SUVR       0.087198
  AMYGDALA_NORM       0.046425
     HIPPO_NORM       0.040125
PRIOR_DIAGNOSIS       0.031213
           PTAU       0.026483
ENTORHINAL_NORM       0.020041
            TAU       0.015334
  ANOMALY_SCORE       0.014705
            AGE       0.012107


**Stage 2.5, Leo's RUL estimate**: a Lasso/RandomForest attribution pass, the same tool
this project's own `risk_factors/1_global.ipynb` already uses, following `06-at`'s own baseline-
before-SHAP progression, applied here to `RUL_YEARS_PRED` rather than `RISK_SCORE`, since that is
the one place in this pipeline a second, independently-trained upstream model's own reasoning has
never been checked at all.


In [61]:
from sklearn.model_selection import train_test_split as sk_split

# Important: this attribution runs on leo_df / leo_out, Leo's OWN dataframe with his OWN
# column names (HIPPO_ICV, ENTORHINAL_ICV, AMYGDALA_ICV -- his own from-scratch loader and
# his own ICV normalization), not on rul_scored / data_full, which is final.csv and uses this
# project's different column names for the same concept (HIPPO_NORM, ENTORHINAL_NORM,
# AMYGDALA_NORM). Section 5 only carried RID/VISCODE2_norm/RUL_YEARS_PRED across that merge
# on purpose, to avoid silently mixing two different normalizations of the same biomarkers
# under one name; leo_feat_cols only exists in leo_df's own column space.
leo_attrib_source = leo_df.copy()
leo_attrib_source['RUL_YEARS_PRED'] = leo_oof_pred

rul_attrib_df = leo_attrib_source.dropna(subset=leo_feat_cols + ['RUL_YEARS_PRED']).copy()
X_rul = rul_attrib_df[leo_feat_cols]
y_rul = rul_attrib_df['RUL_YEARS_PRED']

X_tr, X_te, y_tr, y_te = sk_split(X_rul, y_rul, test_size=0.25, random_state=RANDOM_STATE)

lasso_model, lasso_scaler = du.fit_lasso_baseline(X_tr, y_tr, alpha=0.01)
lasso_weights = du.top_lasso_weights(lasso_model, leo_feat_cols, top_n=len(leo_feat_cols))
print("Lasso attribution on Leo's own RUL_YEARS_PRED (which biomarkers drive his model's output):")
print(f"(Leo's own feature space: {leo_feat_cols})")
lasso_weights


Lasso attribution on Leo's own RUL_YEARS_PRED (which biomarkers drive his model's output):
(Leo's own feature space: ['AGE', 'PTGENDER', 'PTEDUCAT', 'HIPPO_ICV', 'ENTORHINAL_ICV', 'AMYGDALA_ICV', 'CENTILOIDS', 'SUMMARY_SUVR', 'ABETA42', 'TAU', 'PTAU'])


,feature,weight
0,TAU,-0.658166
1,PTAU,0.587766
2,SUMMARY_SUVR,-0.527086
3,HIPPO_ICV,0.255478
4,AMYGDALA_ICV,0.139132
5,ABETA42,0.131468
6,ENTORHINAL_ICV,0.078980
7,PTGENDER,-0.038705
8,AGE,-0.032216
9,PTEDUCAT,-0.011864


**Stage 3, the decision itself**: `recommend_interval` and `recommend_interval_trajectory`
are both closed-form grid searches over a fixed 3-candidate menu, following
`decision_mechanism.py`'s own correct reasoning that a formula this small and fully known gains
nothing from a SHAP-style approximation. The exact same breakdown that script already builds for
the snapshot policy is reused directly rather than reimplemented.


In [62]:
def expected_cost_breakdown(risk, menu=(3, 6, 12), check_cost=CHECK_COST,
                             missed_conversion_cost=MISSED_CONVERSION_COST,
                             reference_interval_months=REFERENCE_INTERVAL_MONTHS):
    rows = []
    for interval in menu:
        routine = (12.0 / interval) * check_cost
        scaled_risk = risk * (interval / reference_interval_months)
        missed = scaled_risk * missed_conversion_cost
        rows.append({'interval': interval, 'routine_cost': routine,
                     'missed_cost': missed, 'total_cost': routine + missed})
    return pd.DataFrame(rows)

example_risk = float(test_score['RISK_SCORE'].median())
print(f"Example: median-risk test patient, RISK_SCORE={example_risk:.3f}")
expected_cost_breakdown(example_risk)


Example: median-risk test patient, RISK_SCORE=0.273


,interval,routine_cost,missed_cost,total_cost
0,3,4.0,1.362821,5.362821
1,6,2.0,2.725642,4.725642
2,12,1.0,5.451284,6.451284


<a id="sec9"></a>

## 9. The Comparison Giorgio's Own Pipeline Never Made: Same Patients, Before Vs After

Giorgio's `fairness_expgrad_report.txt` reports his final numbers next to
`wire_full_pipeline.py`'s own earlier numbers, and both of those next to this project's original
placeholder-score policy numbers in `approach_comparison.csv`, but never on the *same rows*: his
soft-fallback cascade covers 2454 test rows, this project's own original `snapshot_adaptive` row
in `approach_comparison.csv` covers 3665. A total-cost figure computed on 2454 patients is not
directly comparable to one computed on 3665 different patients, the populations differ in which
patients even have enough data to be scored at all. This section fixes that: the *same* patient
set, scored several different ways, so the gain from each individual ingredient (not only "before
vs after everything at once") is visible on a population every version of the score can actually
reach.


In [63]:
placeholder = du.placeholder_risk_score(data_full)
data_full['RISK_SCORE_PLACEHOLDER'] = placeholder

common_ids = scoreable[['RID', 'VISCODE2_norm']].merge(
    data_full.loc[data_full['RISK_SCORE_PLACEHOLDER'].notna(), ['RID', 'VISCODE2_norm']],
    on=['RID', 'VISCODE2_norm'], how='inner',
)
print(f"Common rows (both a placeholder score AND this notebook's enriched RISK_SCORE): {len(common_ids)}")

before_after = scoreable.merge(common_ids, on=['RID', 'VISCODE2_norm'], how='inner')
before_after = before_after.merge(
    data_full[['RID', 'VISCODE2_norm', 'RISK_SCORE_PLACEHOLDER']],
    on=['RID', 'VISCODE2_norm'], how='left',
)

train_ba, test_ba = du.subject_train_test_split(before_after, test_fraction=0.25, random_state=RANDOM_STATE)

def evaluate_on(df, risk_col):
    cost, didi, catch, _ = evaluate_snapshot_policy(df, risk_col=risk_col)
    return cost, didi, catch

before_after_rows = []
for split_name, split_df in [('train', train_ba), ('test', test_ba)]:
    cost_before, didi_before, catch_before = evaluate_on(split_df, 'RISK_SCORE_PLACEHOLDER')
    cost_after, didi_after, catch_after = evaluate_on(split_df, 'RISK_SCORE')
    before_after_rows.append({'split': split_name, 'n_rows': len(split_df),
                               'total_cost_before': cost_before, 'total_cost_after': cost_after,
                               'didi_before': didi_before, 'didi_after': didi_after,
                               'catch_rate_before': catch_before, 'catch_rate_after': catch_after})

before_after_df = pd.DataFrame(before_after_rows)
before_after_df


Common rows (both a placeholder score AND this notebook's enriched RISK_SCORE): 9818


,split,n_rows,total_cost_before,total_cost_after,didi_before,didi_after,catch_rate_before,catch_rate_after
0,train,7569,44928.25,34795.511513,2.981690,4.969441,98.445596,92.746114
1,test,2553,15256.50,11818.105196,3.580203,8.300545,97.457627,94.067797


**This is the number Giorgio's own report never computed**: same patients, snapshot policy
held fixed, only the RISK_SCORE underneath it changed. A real cost improvement here (`_after` <
`_before`) is direct evidence the pipeline helps; a wash or a regression is direct evidence it
does not, on the specific population both scores actually reach. Either answer is worth having
before claiming the pipeline improves on the individual approaches it is built from.


### 9.1 Isolating Each Ingredient's Own Contribution, Same Patients Throughout

The comparison above is two points, placeholder versus everything this notebook adds at once.
That hides *which* addition is doing the work: `RISK_SCORE` already folds ANOMALY_SCORE in
through the soft-fallback cascade (section 6/15), but Leo's `RUL_YEARS_PRED` is never one of the
fallback cascade's own input features (`FEATURE_SETS` in section 4 never includes it), it is only
audited on the side (checkpoint B, section 7, and the Lasso pass in section 8), never merged into
the score itself. So "before vs after" above is really measuring Giorgio's own contribution
alone, not Leo's, and that is worth stating plainly rather than leaving implied.

Three scores, same `before_after` patient set as above, same snapshot policy, same
`train_ba` / `test_ba` split:

- `RISK_SCORE_PLACEHOLDER`: the project's own zero-model baseline (diagnosis + amyloid status
  only, section 9's original comparison).
- `RISK_SCORE_HAZARD_ONLY`: this notebook's own re-fit hazard model, best of the `core` /
  `extended` tiers (no ANOMALY_SCORE feature at all), i.e. Point 2 alone, the way it stood before
  Giorgio's pipeline existed.
- `RISK_SCORE`: the same cascade already used everywhere else in this notebook, `core` /
  `extended` plus their `+anomaly` counterparts, i.e. Point 2 with Giorgio's Point 1 signal
  folded in. Leo's RUL is not a fourth column here, precisely because it was never wired into the
  score to begin with, only into the separate audits above.


In [64]:
def best_tier_score(df, feature_sets, fitted, tier_names):
    # Same soft-fallback pattern as build_risk_score, restricted to a chosen subset of tiers,
    # so "hazard only" and "hazard + anomaly" can each be scored with their own best-tier cascade
    # rather than a single fixed model, matching how RISK_SCORE itself is already built.
    ranked = hazard_results_df[hazard_results_df['tier'].isin(tier_names)] \
        .sort_values('test_auc', ascending=False)
    order = [(row['tier'], row['model']) for _, row in ranked.iterrows()]
    scored = build_risk_score(df, feature_sets, fitted, order)
    return scored['RISK_SCORE']

before_after['RISK_SCORE_HAZARD_ONLY'] = best_tier_score(
    before_after, FEATURE_SETS, hazard_fitted, tier_names=['core', 'extended'])

train_ba, test_ba = du.subject_train_test_split(before_after, test_fraction=0.25, random_state=RANDOM_STATE)

ingredient_rows = []
for split_name, split_df in [('train', train_ba), ('test', test_ba)]:
    for label, col in [('placeholder (no model)', 'RISK_SCORE_PLACEHOLDER'),
                        ('hazard only (Point 2, no anomaly)', 'RISK_SCORE_HAZARD_ONLY'),
                        ('hazard + ANOMALY_SCORE (this notebook\'s RISK_SCORE)', 'RISK_SCORE')]:
        cost, didi, catch = evaluate_on(split_df, col)
        ingredient_rows.append({'split': split_name, 'score': label, 'n_rows': len(split_df),
                                 'total_cost': cost, 'didi': didi, 'catch_rate': catch})

ingredient_df = pd.DataFrame(ingredient_rows)
ingredient_df


/Users/pelle/python_venvs/AII/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/pelle/python_venvs/AII/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


,split,score,n_rows,total_cost,didi,catch_rate
0,train,placeholder (no model),7569,44928.250000,2.981690,98.445596
1,train,"hazard only (Point 2, no anomaly)",7569,35478.016596,5.542059,91.450777
2,train,hazard + ANOMALY_SCORE (this notebook's RISK_S...,7569,34795.511513,4.969441,92.746114
3,test,placeholder (no model),2553,15256.500000,3.580203,97.457627
4,test,"hazard only (Point 2, no anomaly)",2553,11970.440460,7.888891,92.372881
5,test,hazard + ANOMALY_SCORE (this notebook's RISK_S...,2553,11818.105196,8.300545,94.067797


**Reading this table**: compare rows within the same `split`, top to bottom. The step from
`placeholder` to `hazard only` is what Point 2 alone contributes, before Giorgio's Point 1 ever
enters the picture, still on the exact same 9818 patients as every other number in this section.
The step from `hazard only` to `hazard + ANOMALY_SCORE` is what Giorgio's pipeline specifically
adds on top of Point 2, isolated from the placeholder-to-hazard jump it was bundled with in the
first table above. If most of the total gain already happens at the `hazard only` step, that is
evidence Point 2 was already doing most of the work and Giorgio's addition is a smaller marginal
gain, not the headline number, worth knowing before attributing "the pipeline improves things" to
any one contributor specifically.


In [65]:
def cost_metric_factory(risk_col):
    def _metric(df):
        return evaluate_snapshot_policy(df, risk_col=risk_col)[0]
    return _metric

bootstrap_cols = [
    ('placeholder', 'RISK_SCORE_PLACEHOLDER'),
    ('hazard only', 'RISK_SCORE_HAZARD_ONLY'),
    ('hazard + anomaly (RISK_SCORE)', 'RISK_SCORE'),
]
bootstrap_results = {}
for label, col in bootstrap_cols:
    point, low, high, _ = du.bootstrap_ci(test_ba, cost_metric_factory(col), id_col='RID',
                                           n_bootstrap=200, random_state=RANDOM_STATE)
    bootstrap_results[label] = (point, low, high)
    print(f"Total cost, {label:32s}: {point:10.1f}  [{low:.1f}, {high:.1f}]  (95% patient-level bootstrap CI)")

print()
point_before, low_before, high_before = bootstrap_results['placeholder']
point_mid, low_mid, high_mid = bootstrap_results['hazard only']
point_after, low_after, high_after = bootstrap_results['hazard + anomaly (RISK_SCORE)']

def overlap(a_low, a_high, b_low, b_high):
    return not (a_high < b_low or b_high < a_low)

print("placeholder -> hazard only      :",
      "intervals overlap, not clearly distinguishable from noise" if overlap(low_before, high_before, low_mid, high_mid)
      else "intervals do not overlap, unlikely to be noise")
print("hazard only -> hazard + anomaly :",
      "intervals overlap, not clearly distinguishable from noise" if overlap(low_mid, high_mid, low_after, high_after)
      else "intervals do not overlap, unlikely to be noise")
print("placeholder -> hazard + anomaly :",
      "intervals overlap, not clearly distinguishable from noise" if overlap(low_before, high_before, low_after, high_after)
      else "intervals do not overlap, unlikely to be noise")


Total cost, placeholder                     :    15256.5  [14268.7, 16243.1]  (95% patient-level bootstrap CI)
Total cost, hazard only                     :    11970.4  [11197.9, 12896.3]  (95% patient-level bootstrap CI)
Total cost, hazard + anomaly (RISK_SCORE)   :    11818.1  [11064.9, 12722.9]  (95% patient-level bootstrap CI)

placeholder -> hazard only      : intervals do not overlap, unlikely to be noise
hazard only -> hazard + anomaly : intervals overlap, not clearly distinguishable from noise
placeholder -> hazard + anomaly : intervals do not overlap, unlikely to be noise


<a id="sec10"></a>

## 10. Saving Standardized Results

Logged into the same shared `results/decision_support/approach_comparison.csv` every other
`approach_*.ipynb` and Giorgio's own pipeline reports read and write, under a new
`approach_group`, `full_pipeline_reevaluated`, so `comparison.ipynb` can read this notebook's own
results alongside every other approach without a second, differently-shaped file.


In [66]:
for split_name, split_df in [('train', train_ba), ('test', test_ba)]:
    row = before_after_df[before_after_df['split'] == split_name].iloc[0]
    du.append_approach_result(
        RESULTS_DIR, approach=f'pipeline_reevaluated_before_{split_name}',
        approach_group='full_pipeline_reevaluated', split=split_name,
        n_rows=int(row['n_rows']), total_cost=row['total_cost_before'],
        didi=row['didi_before'], catch_rate=row['catch_rate_before'],
        notes='Same patient set as the _after row, placeholder RISK_SCORE, snapshot policy.',
    )
    du.append_approach_result(
        RESULTS_DIR, approach=f'pipeline_reevaluated_after_{split_name}',
        approach_group='full_pipeline_reevaluated', split=split_name,
        n_rows=int(row['n_rows']), total_cost=row['total_cost_after'],
        didi=row['didi_after'], catch_rate=row['catch_rate_after'],
        notes='Same patient set as the _before row, ANOMALY_SCORE + RUL-enriched RISK_SCORE, snapshot policy.',
    )

for _, row in checkpoint_df.iterrows():
    du.append_approach_result(
        RESULTS_DIR, approach=f"fairness_checkpoint_{row['checkpoint'][:1]}",
        approach_group='full_pipeline_reevaluated_fairness', split='test',
        n_rows=int(row['n_rows']), didi=row['didi'], notes=row['checkpoint'],
    )

print(f"Results appended to {os.path.join(RESULTS_DIR, 'approach_comparison.csv')}")


Results appended to ../../results/decision_support/approach_comparison.csv


<a id="sec11"></a>

## 11. Takeaways And Open Items

**What this notebook adds that neither `gio/pipeline_1_2_3` nor the original
`notebooks/decision_support` approach notebooks had on their own**: Leo's RUL estimate as a real,
exported per-row signal (section 5); all four interval policies re-checked against the enriched
RISK_SCORE rather than one assumed winner (section 6); fairness audited at four separate
checkpoints along the chain rather than only the final recommendation (section 7); and, the gap
this notebook exists specifically to close, a same-population before/after comparison with an
honest confidence interval on the cost difference (section 9), which neither Giorgio's own
pipeline reports nor the original approach notebooks ever computed.

**What this notebook deliberately reuses rather than re-runs**, to stay fast: Giorgio's own
ExponentiatedGradient fairness correction and his SHAP-on-a-mixture explainability audit (both
genuinely expensive, both already correct and already saved to disk in `gio/pipeline_1_2_3/`),
and `recommend_interval_forecast`, left out per section 6's own note on its already-documented,
separate calibration problem.

**What is still open after this notebook**, worth deciding before submission, not deciding here:
whether Leo's RUL estimate, once genuinely exported, should be folded directly into RISK_SCORE as
a fifth input tier the way ANOMALY_SCORE already is (a real modeling decision, not something to
default into silently); whether checkpoint A or B in section 7 showing meaningful disparity
changes whether Giorgio's single downstream correction is sufficient, or whether a second
correction upstream is warranted; and whether section 9's before/after difference, once the
bootstrap interval above is read, is large enough and consistent enough to state as a real
improvement in a report to the professor, or narrow enough that the honest claim is "not yet
distinguishable from noise on this sample."
